# Fire Detection Model Training
YOLOv8n fine-tuned on fire, smoke, firefighter, propane tank, and exit sign datasets.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free) or A100 (Pro)
2. Add your Roboflow key: click the 🔑 **Secrets** tab (left sidebar) → add `ROBOFLOW_KEY`

In [ ]:
# Cell 1 — Install dependencies
!pip install -q ultralytics roboflow

In [ ]:
# Cell 2 — Load API key from Colab Secrets
from google.colab import userdata
API_KEY = userdata.get('ROBOFLOW_KEY')
print('API key loaded ✓')

In [ ]:
# Cell 3 — Download all datasets from Roboflow
from roboflow import Roboflow
from pathlib import Path

DATASETS = [
    ("maad-anwar-ertmq",   "smoke-and-fire-detection-a3dtx", 4),
    ("freelance-projects", "propane-cylinder",                1),
    ("juyeon-ko",          "firefighter",                     1),
    ("wsfree",             "exit-cohy8",                      2),
]

rf = Roboflow(api_key=API_KEY)
download_paths = []

for ws, proj, ver in DATASETS:
    print(f'Downloading {ws}/{proj} v{ver}...')
    ds = rf.workspace(ws).project(proj).version(ver).download(
        'yolov8', location=f'/content/raw/{proj}-v{ver}'
    )
    download_paths.append(Path(ds.location))
    print(f'  → {ds.location}')

print('\nAll datasets downloaded ✓')

In [ ]:
# Cell 4 — Merge into a single YOLO dataset
import shutil
import yaml

CLASS_REMAP = {
    'FIRE':         'fire',
    'fire':         'fire',
    'SMOKE':        'smoke',
    'smoke':        'smoke',
    'cylinder':     'propane_tank',
    'firefighters': 'firefighter',
    'firefighter':  'firefighter',
    'exit':         'exit_sign',
}

CLASSES = ['fire', 'smoke', 'firefighter', 'propane_tank', 'exit_sign']

MERGED = Path('/content/dataset')
for split in ('train', 'valid'):
    (MERGED / split / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED / split / 'labels').mkdir(parents=True, exist_ok=True)

totals = {'train': 0, 'valid': 0}

for ds_path in download_paths:
    yaml_path = ds_path / 'data.yaml'
    if not yaml_path.exists():
        print(f'  ⚠️  No data.yaml in {ds_path}')
        continue

    with open(yaml_path) as f:
        meta = yaml.safe_load(f)
    ds_classes = meta.get('names', [])

    for split in ('train', 'valid'):
        img_dir = ds_path / split / 'images'
        lbl_dir = ds_path / split / 'labels'
        if not img_dir.exists():
            continue

        for img_file in img_dir.iterdir():
            if img_file.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            lbl_file = lbl_dir / img_file.with_suffix('.txt').name
            if not lbl_file.exists():
                continue

            new_lines = []
            for line in lbl_file.read_text().strip().splitlines():
                parts = line.split()
                if not parts:
                    continue
                old_idx = int(parts[0])
                if old_idx >= len(ds_classes):
                    continue
                canonical = CLASS_REMAP.get(ds_classes[old_idx], ds_classes[old_idx])
                if canonical not in CLASSES:
                    continue
                new_lines.append(f"{CLASSES.index(canonical)} " + " ".join(parts[1:]))

            if not new_lines:
                continue

            prefix = ds_path.name.replace(' ', '_')
            shutil.copy2(img_file, MERGED / split / 'images' / f'{prefix}_{img_file.name}')
            (MERGED / split / 'labels' / f'{prefix}_{lbl_file.name}').write_text('\n'.join(new_lines))
            totals[split] += 1

    print(f'Merged {ds_path.name}')

# Write data.yaml
data_yaml = {
    'path':  str(MERGED),
    'train': 'train/images',
    'val':   'valid/images',
    'nc':    len(CLASSES),
    'names': CLASSES,
}
with open(MERGED / 'data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f'\nMerge complete → train: {totals["train"]}  valid: {totals["valid"]} images ✓')

In [ ]:
# Cell 5 — Train
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # pretrained COCO weights as starting point

results = model.train(
    data=str(MERGED / 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/runs',
    name='fire_detection',
    exist_ok=True,
)

In [ ]:
# Cell 6 — Evaluate on validation set
best = YOLO('/content/runs/fire_detection/weights/best.pt')
metrics = best.val(data=str(MERGED / 'data.yaml'), device=0)
print(f'mAP50: {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')

In [ ]:
# Cell 7 — Download weights to your machine
from google.colab import files
files.download('/content/runs/fire_detection/weights/best.pt')